# Create Spark Session and Connect Google Drive

In [32]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

spark = SparkSession.builder.appName("MergeDatasets").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(spark.version)

4.0.3


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Complete Dataset

In [3]:
files = [
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-01.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-02.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-03.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-04.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-05.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-06.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-07.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-08.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-09.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-10.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-11.parquet",
    "/content/drive/MyDrive/NYC Yellow Taxi 2025 Dataset/yellow_tripdata_2025-12.parquet"
]

df = spark.read.parquet(*files)

# Phase 1 – Data Understanding

##Number of rows and columns

In [4]:
print(f"Total Rows    : {df.count()}")
print(f"Total Columns : {len(df.columns)}")

Total Rows    : 48722602
Total Columns : 20


##Column Names


In [5]:
for column in df.columns:
  print(column)

VendorID
tpep_pickup_datetime
tpep_dropoff_datetime
passenger_count
trip_distance
RatecodeID
store_and_fwd_flag
PULocationID
DOLocationID
payment_type
fare_amount
extra
mta_tax
tip_amount
tolls_amount
improvement_surcharge
total_amount
congestion_surcharge
Airport_fee
cbd_congestion_fee


##Dataset Schema

In [6]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



##Sample Records

In [7]:
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|1       |2025-05-01 00:07:06 |2025-05-01 00:24:15  |1              |3.7          |1         |N                 |140         |202 

##Summary Statistics


In [8]:
df.describe().show()

+-------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|     PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|           mta_tax|       tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+---------------------+---------------

##Missing Value Analysis

In [9]:
null_df = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_df.show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|0       |0                   |0                    |11611894       |0            |11611894  |11611894          |0           |0   

##Duplicate Record Analysis

In [10]:
total_rows = df.count()

unique_rows = df.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print(f"Total Rows      : {total_rows:,}")
print(f"Unique Rows     : {unique_rows:,}")
print(f"Duplicate Rows  : {duplicate_rows:,}")

Total Rows      : 48,722,602
Unique Rows     : 48,722,601
Duplicate Rows  : 1


##Data Type Analysis

In [11]:
print("="*50)
print("Data Types")
print("="*50)

for name, dtype in df.dtypes:
    print(f"{name:30} {dtype}")

Data Types
VendorID                       int
tpep_pickup_datetime           timestamp_ntz
tpep_dropoff_datetime          timestamp_ntz
passenger_count                bigint
trip_distance                  double
RatecodeID                     bigint
store_and_fwd_flag             string
PULocationID                   int
DOLocationID                   int
payment_type                   bigint
fare_amount                    double
extra                          double
mta_tax                        double
tip_amount                     double
tolls_amount                   double
improvement_surcharge          double
total_amount                   double
congestion_surcharge           double
Airport_fee                    double
cbd_congestion_fee             double


##Distinct Value Analysis

In [12]:
categorical_columns = [
    "VendorID",
    "RatecodeID",
    "payment_type",
    "store_and_fwd_flag"
]

for column in categorical_columns:

    print("\n" + "="*30)
    print(f"Distinct Values of {column}")
    print("="*30)

    df.select(column).distinct().orderBy(column).show()


Distinct Values of VendorID
+--------+
|VendorID|
+--------+
|       1|
|       2|
|       6|
|       7|
+--------+


Distinct Values of RatecodeID
+----------+
|RatecodeID|
+----------+
|      NULL|
|         1|
|         2|
|         3|
|         4|
|         5|
|         6|
|        99|
+----------+


Distinct Values of payment_type
+------------+
|payment_type|
+------------+
|           0|
|           1|
|           2|
|           3|
|           4|
|           5|
+------------+


Distinct Values of store_and_fwd_flag
+------------------+
|store_and_fwd_flag|
+------------------+
|              NULL|
|                 N|
|                 Y|
+------------------+



####Overall Understanding
*   The dataset contains taxi trip, fare, payment, and location details.

*   It has numerical, categorical, and date/time columns.

*   Most columns have complete data, but some contain missing (NULL) values.

*   Some records have invalid or unexpected values.

*   Duplicate records may be present in the dataset.

*   Some columns need data type conversion.













#Phase 2 – Data Cleaning

Per-column Cleaning Notes (Before Cleaning)

| Column / Group | Issue Found | Cleaning Rule | Why |
|----------------|------------|---------------|-----|
| Duplicate Records | Duplicate rows | Remove duplicate records | Avoid repeated taxi trips in analysis |
| VendorID | Incorrect data type | Convert to ByteType | Reduce memory usage and keep valid data type |
| passenger_count | Incorrect data type, values less than 1 | Convert to ByteType and keep values ≥ 1 | Passenger count cannot be zero or negative |
| RatecodeID | NULL values and invalid value (99) | Replace NULL and 99 with 0, convert to ByteType | Standardize invalid rate codes |
| payment_type | Incorrect data type | Convert to ByteType | Improve storage and processing |
| store_and_fwd_flag | NULL values | Replace NULL with "N" | Keep only valid flag values |
| trip_distance | Zero or negative values | Keep values > 0 | Trip distance must be positive |
| fare_amount | Zero or negative values | Keep values > 0 | Fare amount should be positive |
| total_amount | Zero or negative values | Keep values > 0 | Remove invalid payment records |
| Airport_fee | Negative values | Check and keep valid values | Airport fee cannot be negative |
| tpep_pickup_datetime, tpep_dropoff_datetime | Invalid trip order | Keep trips where drop-off > pickup | Ensure valid trip duration |
| All Columns | Missing values and data quality | Verify after cleaning | Ensure dataset is ready for analysis |

##Duplicate Record Removal

In [13]:
df = df.dropDuplicates()
duplicates = df.count() - df.dropDuplicates().count()
print("Remaining duplicates:", duplicates)

Remaining duplicates: 0


##Data bold text Type Conversion

In [14]:
from pyspark.sql.functions import col
from pyspark.sql.types import *

df = (
    df
    .withColumn("VendorID", col("VendorID").cast(ByteType()))
    .withColumn("passenger_count", col("passenger_count").cast(ByteType()))
    .withColumn("RatecodeID", col("RatecodeID").cast(ByteType()))
    .withColumn("PULocationID", col("PULocationID").cast(ShortType()))
    .withColumn("DOLocationID", col("DOLocationID").cast(ShortType()))
    .withColumn("payment_type", col("payment_type").cast(ByteType()))

    .withColumn("trip_distance", col("trip_distance").cast(FloatType()))
    .withColumn("fare_amount", col("fare_amount").cast(FloatType()))
    .withColumn("extra", col("extra").cast(FloatType()))
    .withColumn("mta_tax", col("mta_tax").cast(FloatType()))
    .withColumn("tip_amount", col("tip_amount").cast(FloatType()))
    .withColumn("tolls_amount", col("tolls_amount").cast(FloatType()))
    .withColumn("improvement_surcharge", col("improvement_surcharge").cast(FloatType()))
    .withColumn("congestion_surcharge", col("congestion_surcharge").cast(FloatType()))
    .withColumn("Airport_fee", col("Airport_fee").cast(FloatType()))
    .withColumn("cbd_congestion_fee", col("cbd_congestion_fee").cast(FloatType()))
    .withColumn("total_amount", col("total_amount").cast(FloatType()))
)

In [15]:
df.printSchema()

root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = true)
 |-- Airport_fee: float (nullable = true)
 |-- cbd_congestion_fee: float (nullable = true)



##Missing Value Handling

In [16]:
df = df.na.fill({
    "Airport_fee": 0,
    "congestion_surcharge": 0,
    "cbd_congestion_fee": 0,
})

In [17]:
df.select([
count(when(col(c).isNull(),c)).alias(c)
for c in df.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|       11611894|            0|  11611894|          11611894|           0|    

##RatecodeID Cleaning

In [18]:
df = df.withColumn(
    "RatecodeID",
    when(col("RatecodeID").isNull(),0)
    .otherwise(col("RatecodeID"))
)

df = df.withColumn(
    "RatecodeID",
    when(col("RatecodeID")==99,0)
    .otherwise(col("RatecodeID"))
)

##Store and Forward Flag Cleaning

In [19]:
df = df.na.fill({
    "store_and_fwd_flag": "N"
})

##Validation

In [20]:
df.groupBy("VendorID").count().orderBy("VendorID").show()

df.groupBy("RatecodeID").count().orderBy("RatecodeID").show()

df.groupBy("payment_type").count().orderBy("payment_type").show()

+--------+--------+
|VendorID|   count|
+--------+--------+
|       1| 9586873|
|       2|38575845|
|       6|   23982|
|       7|  535901|
+--------+--------+

+----------+--------+
|RatecodeID|   count|
+----------+--------+
|         0|12416291|
|         1|34292984|
|         2| 1297934|
|         3|  150692|
|         4|  116228|
|         5|  448421|
|         6|      51|
+----------+--------+

+------------+--------+
|payment_type|   count|
+------------+--------+
|           0|11611894|
|           1|31054000|
|           2| 4654344|
|           3|  308147|
|           4| 1094213|
|           5|       3|
+------------+--------+



In [21]:
df.filter(col("store_and_fwd_flag").isNull()).count()

0

In [22]:
df = df.filter(col("passenger_count") >= 1)

In [23]:
df = df.filter(col("trip_distance") > 0)

In [24]:
df = df.filter(col("fare_amount") > 0)

In [25]:
df = df.filter(col("total_amount") > 0)

In [26]:
df.filter(col("Airport_fee")<0).count()

0

In [27]:
df.describe(["Airport_fee"]).show()

+-------+-------------------+
|summary|        Airport_fee|
+-------+-------------------+
|  count|           35604835|
|   mean|0.15662090555959604|
| stddev| 0.5152449982382802|
|    min|                0.0|
|    max|               6.75|
+-------+-------------------+



In [28]:
df.groupBy("store_and_fwd_flag").count().show()

+------------------+--------+
|store_and_fwd_flag|   count|
+------------------+--------+
|                 Y|  100686|
|                 N|35504149|
+------------------+--------+



In [29]:
df = df.filter(
    col("tpep_dropoff_datetime") >
    col("tpep_pickup_datetime")
)

####Cleaning Notes (After Cleaning)

The following cleaning operations were successfully completed:

- Duplicate records were removed.
- Data types were converted to appropriate PySpark types.
- Missing values were handled using suitable business rules.
- Invalid RatecodeID values were standardized.
- Missing values in the Store and Forward Flag were replaced.
- Invalid passenger counts were removed.
- Trips with zero or negative trip distance were removed.
- Records with invalid fare amounts were removed.
- Records with invalid total amounts were removed.
- Pickup and drop-off timestamps were validated.
- The cleaned dataset was verified for consistency and quality.


####Final Verification

In [30]:
print("="*60)
print("FINAL DATA VERIFICATION")
print("="*60)

print("Rows :",df.count())
print("Columns :",len(df.columns))

print("Duplicates :",df.count()-df.dropDuplicates().count())

df.select([
count(when(col(c).isNull(),c)).alias(c)
for c in df.columns
]).show()

df.printSchema()

FINAL DATA VERIFICATION
Rows : 35075557
Columns : 20
Duplicates : 0
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|         

#Save Clean Dataset

In [31]:
output_path = "/content/drive/MyDrive/NYC_Taxi_2025_Cleaned"

df.write \
.mode("overwrite") \
.parquet(output_path)

print("Dataset saved successfully!")

Dataset saved successfully!
